In [ ]:
!pip install numpy opencv-python matplotlib ultralytics scikit-image scipy torch torchvision skan 

In [ ]:
from ultralytics import YOLO
model = YOLO("trained_yolo.pt")  # load a custom model

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# List of input images
img_paths = [
    "test1.png",
    "test2.png",
    "test3.png",
    "test4.png"
]

for img_path in img_paths:

    # Read image
    img = cv2.imread(img_path)

    if img is None:
        print(f"Could not read {img_path}")
        continue

    # Get filename without extension
    filename = os.path.splitext(os.path.basename(img_path))[0]

    # Padding
    pad = 20
    img = cv2.copyMakeBorder(
        img,
        pad, pad, pad, pad,
        cv2.BORDER_REFLECT
    )
    # BGR -> LAB
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

     # Split channels
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8,8))
    l_clahe = clahe.apply(l)
    # Merge channels
    lab_clahe = cv2.merge((l_clahe, a, b))
    # LAB -> BGR
    result = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)
    
    # Sharpening
    sharpening_kernel = np.array([
        [-1, -2, -1],
        [-2, 13, -2],
        [-1, -2, -1]
    ])

    result = cv2.filter2D(result, -1, sharpening_kernel)
    
    result = result[
    pad:-pad,
    pad:-pad]
    # Save outputs
    cv2.imwrite(f"{filename}_sharpened.png", result)

    # Convert for display
    img = img[
    pad:-pad,
    pad:-pad]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    sharpened_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

    # Display
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.title(f"{filename} - Original")
    plt.imshow(img_rgb)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title(f"{filename} - Sharpened")
    plt.imshow(sharpened_rgb)
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"Processed {filename}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os
from skimage.morphology import remove_small_objects

img_paths = [
    "test1_sharpened.png",
    "test2_sharpened.png",
    "test3_sharpened.png",
    "test4_sharpened.png"
]

for img_path in img_paths:

    img_bgr = cv2.imread(img_path)

    if img_bgr is None:
        print(f"Could not read {img_path}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Resize image for visualization (keep aspect ratio if required)
    img_resized = cv2.resize(
        img_rgb,
        (1024, 1024),
        interpolation=cv2.INTER_CUBIC
    )

    # YOLO prediction
    results = model.predict(
        source=img_path,
        conf=0.1,
        save=False
    )

    if results[0].masks is None:
        print(f"No crack detected in {img_path}")
        continue

    # Extract masks
    masks = results[0].masks.data.cpu().numpy()

    # Combine all crack masks
    mask_output = np.any(masks > 0.5, axis=0)

    # Remove tiny noise
    mask_output = remove_small_objects(
        mask_output.astype(bool),
        min_size=10
    )

    mask_output = mask_output.astype(np.uint8)

    # Resize mask using nearest neighbour
    mask_output = cv2.resize(
        mask_output,
        (1024, 1024),
        interpolation=cv2.INTER_NEAREST
    )

    filename = os.path.splitext(os.path.basename(img_path))[0]

   # Save visualization as PDF

    fig = plt.figure(figsize=(10, 5), dpi=300)

    plt.subplot(1, 2, 1)
    plt.title(filename)
    plt.imshow(img_resized)
    plt.axis("off")


    plt.subplot(1, 2, 2)
    plt.title("Predicted Crack Mask")
    plt.imshow(mask_output, cmap="gray")
    plt.axis("off")


    plt.tight_layout()

    # High quality PDF
    plt.savefig(
        f"{filename}_result.pdf",
        bbox_inches="tight",
        pad_inches=0
    )

    # High quality PNG also
    plt.savefig(
        f"{filename}_result.png",
        dpi=600,
        bbox_inches="tight",
        pad_inches=0
    )

    plt.show()
    plt.close(fig)


    # -------------------------------
    # Save binary mask separately
    # -------------------------------

    mask_to_save = (mask_output * 255).astype(np.uint8)

    cv2.imwrite(
        f"{filename}_mask.png",
        mask_to_save
    )


    print(f"Processed {filename}")

In [ ]:
import numpy as np
from scipy.ndimage import label as ndi_label
from scipy.ndimage import distance_transform_edt
from skimage.measure import regionprops
from skimage.morphology import skeletonize
from skan import Skeleton


def measure_cracks(mask):

    # Label connected components
    mask = np.squeeze(mask)
    labs, num_features = ndi_label(mask > 0)

    out = []

    MIN_AREA = 10   # Ignore tiny blobs

    for region in regionprops(labs):

        if region.area < MIN_AREA:
            continue

        comp = (labs == region.label)

        sk = skeletonize(comp)

        if np.sum(sk) < 2:
            continue

        try:
            skeleton = Skeleton(sk)
            lengths = skeleton.path_lengths()  # calculate lengths of all paths/branches in the skeleton

            if len(lengths) == 0:
                continue

            # Get indices of the 5 longest paths
            top5_paths = np.argsort(lengths)[::-1][:3]

            points = []
            orientations = []

            for path_idx in top5_paths:

                coords = skeleton.path_coordinates(path_idx)

                y1, x1 = coords[0]      # first endpoint
                y2, x2 = coords[-1]     # last endpoint

                dx = x2 - x1
                dy = y2 - y1

                angle = np.degrees(np.arctan2(dy, dx)) % 180

                points.append([[float(x1), float(y1)], [float(x2), float(y2)]])
                orientations.append(round(float(angle), 2))

            L_px = lengths.sum()  # total skeleton length

        except Exception:
            continue

        if L_px == 0:
            continue

        D = distance_transform_edt(comp)
        W_px = 2 * D[sk]

        out.append({
            'label': region.label,
            'length_px': round(L_px, 2),
            'mean_width_px': round(float(W_px.mean()), 2),
            'max_width_px': round(float(W_px.max()), 2),
            'orientation_deg': orientations,
            'points': points,
            'bbox': region.bbox
        })

    return out


def format_crack_summary(measurements):

    lines = []

    for m in measurements:

        lines.append(
            f"Crack {m['label']}: "
            f"Length = {m['length_px']} px, "
            f"Average Width = {m['mean_width_px']} px, "
            f"Maximum Width = {m['max_width_px']} px, "
            f"Orientations = {m['orientation_deg']}"
        )

    return lines

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

mask_paths=[
    "test1_sharpened_mask.png",
    "test2_sharpened_mask.png",
    "test3_sharpened_mask.png",
    "test4_sharpened_mask.png"
]

crack_prompts={}
for mask_path in mask_paths:

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        print(f"Could not read {mask_path}")
        continue

    filename = os.path.basename(mask_path).replace("_sharpened_mask.png", ".png")
    crack_prompts[filename] = []

    mask = (mask > 0).astype(np.uint8)

    measurements = measure_cracks(mask)
    summary = format_crack_summary(measurements)

    print("="*60)
    print(filename)
    print("="*60)

    for line in summary:
        print(line)


    fig, ax = plt.subplots(figsize=(6,6), dpi=300)

    ax.imshow(mask, cmap="gray")


    for m in measurements:

        minr, minc, maxr, maxc = m["bbox"]

        points = np.asarray(
            m["points"],
            dtype=int
        ).tolist()


        rect = Rectangle(
            (minc, minr),
            maxc-minc,
            maxr-minr,
            edgecolor="red",
            linewidth=1,
            fill=False
        )

        ax.add_patch(rect)


        ax.text(
            minc,
            minr-3,
            str(m["label"]),
            color="yellow",
            fontsize=10,
            bbox=dict(
                facecolor="black",
                alpha=0.7
            )
        )


        crack_prompts[filename].append({
            "box": np.array([minc, minr, maxc, maxc]),
            "points": points
        })


    ax.set_title(filename)
    ax.axis("off")

print(crack_prompts)

In [ ]:
!if not exist segment-anything-2 git clone https://github.com/facebookresearch/sam2.git segment-anything-2
%cd segment-anything-2
!pip install -e .

In [ ]:
!wget -O sam2_hiera_small.pt "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt"

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import remove_small_objects


from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_checkpoint = "checkpoints/sam2_hiera_small.pt"
model_cfg = "configs/sam2/sam2_hiera_s.yaml"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
predictor = SAM2ImagePredictor(sam2_model)

state_dict = torch.load(
    "sam2_crack_finetuned.pth",
    map_location=device,
    weights_only=True
)
predictor.model.load_state_dict(state_dict)
predictor.model.eval()


In [ ]:
local_base_dir = r"C:\Path\To\Your\Local\Images" # mention the path where your images are stored

In [ ]:
for img_name in crack_prompts.keys():
    image_path = os.path.join(local_base_dir, img_name)
    img_bgr = cv2.imread(image_path)

    if img_bgr is None:
        print(f"Could not read {image_path}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    img_resized = cv2.resize(
        img_rgb,
        (1024, 1024)
    )

    predictor.set_image(img_resized)

    # Final merged mask
    final_mask = np.zeros((1024, 1024), dtype=bool)

   
    # Run SAM for every detected crack
    for prompt in crack_prompts[img_name]:

        input_point = np.array(
            [pt for pair in prompt["points"] for pt in pair],
            dtype=np.float32
        )
        
        input_box = np.array(
            prompt["box"],
            dtype=np.float32
        )

        input_label = np.ones(
            len(input_point),
            dtype=np.int32
        )

        with torch.no_grad():
            masks, scores, logits = predictor.predict(
                point_coords=input_point,
                point_labels=input_label,
                box=input_box,
                multimask_output=True
            )

            best = np.argmax(scores)

            masks, scores, logits = predictor.predict(
                point_coords=input_point,
                point_labels=input_label,
                box=input_box,
                mask_input=logits[best][None, :, :],
                multimask_output=False
            )

        if masks is None or len(masks) == 0:
            continue

        best_mask = masks[0] > 0.5

        best_mask = remove_small_objects(
            best_mask.astype(bool),
            min_size=10
        )

        final_mask |= best_mask

    # Final prediction
    pred = final_mask.astype(np.uint8)

    pred = cv2.resize(
        pred,
        (1024, 1024),
        interpolation=cv2.INTER_NEAREST
    )
    
    # Create SAM overlay & Binary visualization
    overlay = img_resized.copy()
    
    # Color the predicted crack region (Green [0, 255, 0] for RGB)
    overlay[pred == 1] = [0, 255, 0]
    
    # Blend overlay with original image
    annotated = cv2.addWeighted(
        img_resized,
        0.7,
        overlay,
        0.3,
        0
    )
    
    # Get base filename for titles and saving
    base_name = os.path.splitext(img_name)[0]
    
    # Display the plot inline (Side-by-Side: Overlay and Binary Mask)
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    axes[0].imshow(annotated)
    axes[0].set_title(f"SAM Overlay - {base_name}")
    axes[0].axis("off")
    
    # Use cmap='gray' to show the binary mask in black and white properly
    axes[1].imshow(pred, cmap='gray')
    axes[1].set_title(f"Binary Mask - {base_name}")
    axes[1].axis("off")
    
    plt.tight_layout()
   
    # Save raw binary mask output to disk (as a standard 0-255 image)
    # Construct the full save path using the directory you read from
    save_path = os.path.join(local_base_dir, f"{base_name}_sam.png")
    
    # Save raw binary mask output to that local disk folder
    cv2.imwrite(save_path, (pred * 255).astype(np.uint8))
    print(f"Saved mask to: {save_path}")

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

mask_paths = [os.path.join(local_base_dir, f) for f in ["test1_sam.png", "test2_sam.png"]]

for mask_path in mask_paths:

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        print(f"Could not read {mask_path}")
        continue

    filename = os.path.splitext(os.path.basename(mask_path))[0]

    mask = (mask > 0).astype(np.uint8)

    measurements = measure_cracks(mask)
    summary = format_crack_summary(measurements)

    print("="*60)
    print(filename)
    print("="*60)

    if len(summary) == 0:
        print("No cracks detected.")
    else:
        for line in summary:
            print(line)

    fig, ax = plt.subplots(figsize=(6,6))
    ax.imshow(mask, cmap="gray")
  
    for m in measurements:

        minr, minc, maxr, maxc = m["bbox"]

        rect = Rectangle(
            (minc, minr),
            maxc-minc,
            maxr-minr,
            edgecolor="red",
            linewidth=0.5,
            fill=False
        )

        ax.add_patch(rect)

        ax.text(
        minc,
        minr-3,
        str(m["label"]),
        color="yellow",
        bbox=dict(facecolor="black", alpha=0.7)
    )
    ax.set_title(filename)
    ax.axis("off")

    plt.show()